In [21]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [22]:
RAW_PATH = "/Users/student/Downloads/archive/PlayerStatistics.csv"
df = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Raw rows: {len(df):,}  |  Columns: {df.shape[1]}")

Raw rows: 1,668,860  |  Columns: 40


In [23]:
df['gameDate']   = pd.to_datetime(df['gameDate'], errors='coerce')
df['numMinutes'] = pd.to_numeric(df['numMinutes'], errors='coerce')
df['year']       = df['gameDate'].dt.year
df['decade']     = (df['year'] // 10) * 10

In [24]:
regular_season = df[df['gameType'] == 'Regular Season'].copy()
print(f"\nAfter keeping Regular Season: {len(regular_season):,} rows.")


After keeping Regular Season: 1,502,661 rows.


In [25]:
# Remove DNP rows
regular_season = regular_season[regular_season['comment'].isna()].copy()

# Remove players without recorded minutes
regular_season = regular_season[regular_season['numMinutes'].notna()].copy()

# Remove players who didn't play
regular_season = regular_season[regular_season['numMinutes'] > 0].copy()
print(f"After removing DNP and zero-minute rows: {len(regular_season):,} rows.")

After removing DNP and zero-minute rows: 1,258,827 rows.


In [26]:
# Aggregate game-by-game data to season level data

STAT_COLS = [
    'points', 'assists', 'blocks', 'steals',
    'fieldGoalsAttempted', 'fieldGoalsMade', 'fieldGoalsPercentage',
    'threePointersAttempted', 'threePointersMade', 'threePointersPercentage',
    'freeThrowsAttempted', 'freeThrowsMade', 'freeThrowsPercentage',
    'reboundsDefensive', 'reboundsOffensive', 'reboundsTotal',
    'foulsPersonal', 'turnovers', 'plusMinusPoints', 'numMinutes',
]

In [27]:
# Most common position listed for a player in that season (since some players are listed in different positions depending on game)
def mode_position(s):
    positions = s.dropna()
    return positions.mode()[0] if len(positions) else np.nan

In [28]:
agg = regular_season.groupby(['personId', 'year']).agg(
    firstName        = ('firstName',  'first'),
    lastName         = ('lastName',   'first'),
    decade           = ('decade',     'first'),
    games_played     = ('gameId',     'count'),
    total_minutes    = ('numMinutes', 'sum'),
    avg_minutes      = ('numMinutes', 'mean'),
    position         = ('startingPosition', mode_position),
    **{col: (col, 'mean') for col in STAT_COLS if col != 'numMinutes'},
).reset_index()

In [29]:
print(f"\nPlayer-season rows before noise filter: {len(agg):,}")


Player-season rows before noise filter: 28,383


In [30]:
# Remove noisy players, that is, players who made minor contributions
# All players included in the final dataset must meet two criterium:
# 1. ≥ 20 games played in the season (no short contract or injured players)
# 2. ≥ 15 average minutes per game (meaningful contribution in game rotation)
MIN_GAMES         = 20
MIN_AVG_MINUTES   = 15

In [31]:
mask = (
    (agg['games_played']  >= MIN_GAMES) &
    (agg['avg_minutes']   >= MIN_AVG_MINUTES)
)
clean = agg[mask].copy()
print(f"After noise filter (games≥{MIN_GAMES}, avg_min≥{MIN_AVG_MINUTES}): {len(clean):,} player-seasons")
print(f"Unique players retained: {clean['personId'].nunique():,}")

After noise filter (games≥20, avg_min≥15): 16,255 player-seasons
Unique players retained: 2,606


In [32]:
# Raw positions: G (guard), F (forward), C (center), NaN
# NaN position = player only appeared as non-starter in that season.

print(f"\nPosition distribution (after filter):")
print(clean['position'].value_counts(dropna=False))


Position distribution (after filter):
position
NaN    6818
G      5375
F      3099
C       963
Name: count, dtype: int64


In [33]:
print(f"\nDecade distribution (after filter):")
print(clean.groupby('decade')['personId'].count().sort_index())


Decade distribution (after filter):
decade
1950.0     102
1960.0     663
1970.0    1824
1980.0    2354
1990.0    2733
2000.0    2993
2010.0    3243
2020.0    2343
Name: personId, dtype: int64


In [34]:
# Compile final dataset

OUTPUT_COLS = [
    'personId', 'firstName', 'lastName', 'year', 'decade',
    'games_played', 'total_minutes', 'avg_minutes',
    'position',
    
    # counting stats (per game)
    'points', 'assists', 'blocks', 'steals', 'turnovers',
    'reboundsTotal', 'reboundsOffensive', 'reboundsDefensive',
    'foulsPersonal', 'plusMinusPoints',
    
    # shooting
    'fieldGoalsAttempted', 'fieldGoalsMade', 'fieldGoalsPercentage',
    'threePointersAttempted', 'threePointersMade', 'threePointersPercentage',
    'freeThrowsAttempted', 'freeThrowsMade', 'freeThrowsPercentage',
]
clean_final = clean[OUTPUT_COLS].reset_index(drop=True)

# Round all numeric stat columns to 2 decimal places
stat_cols = clean_final.select_dtypes(include='number').columns.difference(
    ['personId', 'year', 'decade', 'games_played', 'allstar_proxy']
)
clean_final[stat_cols] = clean_final[stat_cols].round(2)

print(f"Raw rows: {len(clean_final):,}  |  Columns: {clean_final.shape[1]}")
print(clean_final.columns)

Raw rows: 16,255  |  Columns: 28
Index(['personId', 'firstName', 'lastName', 'year', 'decade', 'games_played',
       'total_minutes', 'avg_minutes', 'position', 'points', 'assists',
       'blocks', 'steals', 'turnovers', 'reboundsTotal', 'reboundsOffensive',
       'reboundsDefensive', 'foulsPersonal', 'plusMinusPoints',
       'fieldGoalsAttempted', 'fieldGoalsMade', 'fieldGoalsPercentage',
       'threePointersAttempted', 'threePointersMade',
       'threePointersPercentage', 'freeThrowsAttempted', 'freeThrowsMade',
       'freeThrowsPercentage'],
      dtype='object')


In [35]:
OUT_PATH = "/Users/student/Downloads/Math 2320 Final Project/nba_cleaned.csv"
clean_final.to_csv(OUT_PATH, index=False)
print(f"\n Cleaned dataset saved → {OUT_PATH}")
print(f"  Shape: {clean_final.shape}")


 Cleaned dataset saved → /Users/student/Downloads/Math 2320 Final Project/nba_cleaned.csv
  Shape: (16255, 28)
